# Stacking Ensemble Final

Template eksperimen Current Stress dengan tracking MLflow yang konsisten.

In [ ]:
from pathlib import Path
import tempfile
import mlflow
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

from mlflow_utils import (
    DATASET_NAME,
    DATASET_VERSION,
    EXPERIMENT_NAME,
    RANDOM_STATE,
    TEST_SIZE,
    build_preprocessor,
    evaluate_classification,
    load_dataset,
    log_and_register_model,
    log_classification_artifacts,
    log_dataset_inputs,
    log_run_metadata,
    select_features,
    set_seeds,
)


In [ ]:
set_seeds(RANDOM_STATE)

repo_root = Path.cwd().resolve()
for parent in [repo_root, *repo_root.parents]:
    if (parent / ".git").exists():
        repo_root = parent
        break

tracking_dir = repo_root / "mlruns"
tracking_dir.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f"file:{tracking_dir.resolve().as_posix()}")
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

_, X, y = load_dataset()
X = select_features(X, feature_group="all")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

num_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]
preprocessor = build_preprocessor(num_cols, cat_cols)

base_estimators = [
    ("lr", LogisticRegression(max_iter=1500, random_state=RANDOM_STATE)),
    ("dt", DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=8, min_samples_split=4)),
    ("rf", RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=300, n_jobs=-1, class_weight="balanced")),
]
stacking = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=1500, random_state=RANDOM_STATE),
    cv=5,
)
pipeline = Pipeline([("preprocessor", preprocessor), ("classifier", stacking)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="f1_weighted", n_jobs=-1)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)
metrics = evaluate_classification(y_test, y_pred, y_proba)

run_name = "Stacking Ensemble Final"
description = "Stacking ensemble final; base=[LR, RF, DT]; meta=LR; dataset=current_stress_v1"
with mlflow.start_run(run_name=run_name) as run:
    tags = {"project":"nostressia","task":"current-stress","features":"all","model":"Stacking",
            "dataset_name":DATASET_NAME,"dataset_version":DATASET_VERSION,"split":"80/20","random_state":str(RANDOM_STATE)}
    log_run_metadata(tags, description)
    log_dataset_inputs(X_train, y_train, X_test, y_test)
    mlflow.log_params({"model_type":"StackingClassifier", "base_models":"LR,DT,RF", "meta_model":"LR"})
    mlflow.log_metric("cv_f1_weighted_mean", float(np.mean(cv_scores)))
    mlflow.log_metric("cv_f1_weighted_std", float(np.std(cv_scores)))
    mlflow.log_metrics(metrics)

    with tempfile.TemporaryDirectory() as td:
        artifact_dir = Path(td)
        log_classification_artifacts(y_test, y_pred, y_proba, labels=sorted(y.unique()), artifact_dir=artifact_dir)
        mlflow.log_artifacts(str(artifact_dir), artifact_path="evaluation")

    log_and_register_model(pipeline, X_train, model_name="CurrentStress_StackingEnsemble")

print(metrics)
print(f"Run selesai: {run.info.run_id}")
